# Qwen3-VL Shared Domain Base — Training v2 (overnight, self-resuming)

**Round 2.** Round 1's quantitative eval (`Stage4_QwenDomainBase_Eval.ipynb`) showed:
relation 90% (works), typed-summary F1 0.36 (works), counting collapsed to "always 0"
(learned the zero-skewed prior), and the list-all-tags OCR task was **net-destructive**
(tag recall 39% → 2% — it overwrote the base's native reading with degenerate patterns).

**Changes in v2 (all data-side; the training machinery was proven fine):**
1. **Count rebalanced + de-templated** — zero-count tiles capped at ~15%, six target phrasings.
2. **List-all-tags OCR dropped, replaced with single-tag reading** — tight crop around one
   confident Tesseract word → "what does this label say?" Trains the skill Stages 2/5/13
   actually use, on an easy, learnable footprint.
3. **Relation upweighted ~2×** (500 patches) — it serves the two stages with real ground truth.
4. **From-scratch LoRA** (not resumed from v1 — v1's pathologies would fight the fix).
5. Edge-sliver tiles filtered (v1's aspect-ratio crashes), `torchao` uninstall baked in,
   checkpoints go to `v2/` in the same HF repo so v1 stays comparable.

Same resilience as v1: HF-only data (no Drive/Google auth), checkpoint every 200 steps
pushed to HF, exact epoch/step auto-resume on rerun, per-step failure skip, clean
time-budget stop. **Use Colab Pro+ background execution when you Run All.**

## 1. Config — paste your HF token

In [ ]:
HF_TOKEN = "paste-your-hf-token-here"

DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
CKPT_SUBDIR = "v2"                    # keeps v1 checkpoints intact in the same repo
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

N_EPOCHS = 3
LR = 1e-4
CHECKPOINT_EVERY_N_STEPS = 200
TIME_BUDGET_HOURS = 11
MAX_ZERO_COUNT_FRACTION = 0.15        # v1 lesson: unchecked zero-tiles taught "always answer 0"
MAX_KAGGLE_EXAMPLES = 1000
MAX_RELATION_PATCHES = 500            # ~2x v1 (250) - relation serves the real-GT stages
RELATION_PAIRS_PER_PATCH = 12
MAX_TAG_READ_EXAMPLES = 1200
TESS_MIN_CONF = 60                    # only train tag-reading on words Tesseract is confident about

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here", "Paste your HF token above"

## 2. Install + GPU check (torchao removal baked in — v1 lesson)

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q peft pytesseract huggingface_hub
!pip uninstall -y torchao -q

import torch
assert torch.cuda.is_available(), "No GPU - set Runtime type first"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 3. Data from HF → local disk

In [ ]:
import zipfile, time
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)
t0 = time.time()
for fname, extract_to in [
    ("gupta_pid/PID_Dataset.zip", DATA / "gupta"),
    ("kaggle_pid_symbols/kaggle_pid_symbols.zip", DATA / "kaggle"),
    ("pid2graph/PID2Graph.zip", DATA / "pid2graph"),
]:
    if extract_to.exists() and any(extract_to.iterdir()):
        print(f"{extract_to} already extracted"); continue
    zp = hf_hub_download(repo_id=DATA_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(extract_to)
    print(f"extracted {fname} ({time.time()-t0:.0f}s)")

GUPTA_RAW = DATA / "gupta" / "PID_Dataset" / "0__raw_data"
KAGGLE_IMAGES = DATA / "kaggle" / "archive" / "images (3)"
KAGGLE_LABELS = DATA / "kaggle" / "archive" / "labels (2)"
PID2GRAPH_OPEN100 = DATA / "pid2graph" / "PID2Graph" / "Patched" / "PID2Graph OPEN100"
for p in [GUPTA_RAW / "sheets" / "train", KAGGLE_IMAGES, KAGGLE_LABELS, PID2GRAPH_OPEN100]:
    assert p.exists(), f"missing: {p}"
print("data ready")

## 4. Class names (inlined)

In [ ]:
CLASS_NAMES = {
    "0": "Not_used", "1": "Gate_Valve", "2": "Ball_Valve", "3": "Globe_valve_NO",
    "4": "Gate_valve_NO", "5": "Globe_valve_NO", "6": "Butterfly_valve", "7": "Plug_valve",
    "8": "Check_valve", "9": "Diaphragm_valve", "10": "Needle_valve",
    "11": "Half_Filled_Gate_Valve", "12": "Gate_Valve_NC", "13": "Globle_valve_NC",
    "14": "Control_Valve", "15": "Rotary_Valve", "16": "Ball_valve_NC", "17": "Paddle_blind",
    "18": "Spectacle_blind_Closed", "19": "Spectacle_blind_Open", "20": "Reducer",
    "21": "Flange_or_Nozzle", "22": "Rupture_disk", "23": "Pipe_Insulation_or_Tracing",
    "24": "Flow_Arrow", "25": "sight_glass", "26": "Instrument_Field", "27": "Instrument_Field",
    "28": "Instrument_Panel", "29": "Instrument_Aux_Panel", "30": "box",
    "31": "Instrument_Panel", "32": "box",
}

## 5. Build the v2 4-task pool

Counting (rebalanced, 6 phrasings) + single-tag reading (NEW) + typed summary + connectivity
(upweighted). Zero-count capping and phrase diversity are the direct fixes for v1's failures.

In [ ]:
import json, random
import xml.etree.ElementTree as ET
import pytesseract
from PIL import Image
from collections import Counter

Image.MAX_IMAGE_PIXELS = None
TILE_SIZE, OVERLAP = 1024, 205
MIN_TILE_SIDE = 64   # v1 lesson: edge slivers crash Qwen's processor (aspect ratio > 200)

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            if (x1 - x0) >= MIN_TILE_SIDE and (y1 - y0) >= MIN_TILE_SIDE:
                tiles.append((x0, y0, x1, y1))
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return parts[0], [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def boxes_intersect(a, b):
    return a[0] < b[2] and a[2] > b[0] and a[1] < b[3] and a[3] > b[1]

rng = random.Random(0)

# --- Task 1: symbol counting - rebalanced + de-templated ---
COUNT_PROMPTS = [
    "How many distinct symbols (valves, instruments, fittings, etc.) are visible in this P&ID tile?",
    "Count the equipment symbols in this P&ID tile.",
    "How many P&ID symbols does this tile contain?",
]
def count_target(n):
    """Six phrasings so the sentence template can't be learned independent of the image.
    Every phrasing contains the bare number so downstream parsing stays trivial."""
    if n == 0:
        return rng.choice([
            "There are 0 symbols visible in this tile.",
            "This tile contains 0 symbols - it looks like empty margin or plain piping.",
            "I count 0 equipment symbols here.",
        ])
    return rng.choice([
        f"There are {n} symbols visible in this tile.",
        f"I count {n} distinct equipment symbols in this tile.",
        f"This tile contains {n} symbols.",
        f"Looking across the tile, I find {n} symbols in total.",
        f"{n} equipment symbols are visible here.",
        f"The tile shows {n} distinct P&ID symbols.",
    ]) if n != 1 else rng.choice([
        "There is 1 symbol visible in this tile.",
        "I count exactly 1 equipment symbol in this tile.",
        "This tile contains just 1 symbol.",
    ])

# --- Task 2 (NEW): single-tag reading - replaces v1's destructive list-all-tags OCR ---
TAG_READ_PROMPTS = [
    "What does the text label in this crop say?",
    "Read the tag shown in this image crop.",
    "What is written in this label?",
]
def tag_read_target(tag):
    return rng.choice([
        f"The label reads: {tag}",
        f"It says {tag}.",
        f"The tag is {tag}.",
    ])
TAG_CROP_MARGIN = 25

def is_tag_like(text):
    t = text.strip()
    return len(t) >= 2 and (any(c.isdigit() for c in t) or "-" in t) and not t.isspace()

def build_gupta_tasks():
    count_examples, tag_examples = [], []
    zero_examples = []
    for sheet_path in sorted((GUPTA_RAW / "sheets" / "train").glob("*.*")):
        label_path = GUPTA_RAW / "labels" / "train" / f"{sheet_path.stem}.txt"
        img = Image.open(sheet_path).convert("RGB")
        W, H = img.size
        orig_boxes = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    orig_boxes.append(yolo_line_to_xyxy(line, W, H)[1])
        for tbox in compute_tile_grid(W, H):
            x0, y0, x1, y1 = tbox
            n_boxes = sum(1 for b in orig_boxes if boxes_intersect(b, tbox))
            ex = {"task": "count", "image_path": str(sheet_path), "crop": list(tbox),
                  "prompt": rng.choice(COUNT_PROMPTS), "target_text": count_target(n_boxes)}
            (zero_examples if n_boxes == 0 else count_examples).append(ex)

            # single-tag reading: confident Tesseract words, tight crops (sheet coords)
            if len(tag_examples) < MAX_TAG_READ_EXAMPLES:
                crop_img = img.crop(tbox)
                data = pytesseract.image_to_data(crop_img, output_type=pytesseract.Output.DICT)
                cands = [i for i in range(len(data["text"]))
                         if is_tag_like(data["text"][i]) and int(data["conf"][i]) >= TESS_MIN_CONF]
                rng.shuffle(cands)
                for i in cands[:2]:
                    wx0 = x0 + data["left"][i] - TAG_CROP_MARGIN
                    wy0 = y0 + data["top"][i] - TAG_CROP_MARGIN
                    wx1 = x0 + data["left"][i] + data["width"][i] + TAG_CROP_MARGIN
                    wy1 = y0 + data["top"][i] + data["height"][i] + TAG_CROP_MARGIN
                    wx0, wy0 = max(0, wx0), max(0, wy0)
                    wx1, wy1 = min(W, wx1), min(H, wy1)
                    if wx1 - wx0 < 20 or wy1 - wy0 < 12:
                        continue
                    tag = data["text"][i].strip()
                    tag_examples.append({"task": "tag_read", "image_path": str(sheet_path),
                                         "crop": [wx0, wy0, wx1, wy1],
                                         "prompt": rng.choice(TAG_READ_PROMPTS),
                                         "target_text": tag_read_target(tag)})
    # cap zero-count tiles (v1: unchecked zeros taught "always answer 0")
    rng.shuffle(zero_examples)
    max_zeros = int(len(count_examples) * MAX_ZERO_COUNT_FRACTION / (1 - MAX_ZERO_COUNT_FRACTION))
    count_examples.extend(zero_examples[:max_zeros])
    return count_examples, tag_examples

# --- Task 3: typed summary (unchanged - it worked) ---
TYPED_SUMMARY_PROMPT = "What types of symbols are visible in this P&ID tile, and how many of each?"
def build_kaggle_examples(max_images):
    examples = []
    for lbl in sorted(KAGGLE_LABELS.glob("*.txt"))[:max_images]:
        lines = [l for l in lbl.read_text().splitlines() if l.strip()]
        img_path = KAGGLE_IMAGES / f"{lbl.stem}.jpg"
        if not lines or not img_path.exists():
            continue
        counts = Counter(l.split()[0] for l in lines)
        parts = [f"{CLASS_NAMES.get(cid, 'class_' + cid)} x{n}" for cid, n in counts.most_common()]
        examples.append({"task": "typed_summary", "image_path": str(img_path), "crop": None,
                         "prompt": TYPED_SUMMARY_PROMPT,
                         "target_text": "This tile contains: " + ", ".join(parts) + "."})
    return examples

# --- Task 4: connectivity (unchanged mechanics, upweighted) ---
RELATION_PROMPT_TEMPLATE = (
    "In this P&ID crop there is a symbol at [{ax0}, {ay0}, {ax1}, {ay1}] and another at "
    "[{bx0}, {by0}, {bx1}, {by1}] (pixel coordinates, top-left origin). "
    "Are these two symbols directly connected to each other?"
)
MAX_PAIR_SPAN_PX = 1400
CROP_MARGIN = 80

def parse_graphml(path):
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {keymap.get(d.get("key"), ""): d.text for d in node.findall("g:data", ns)}
        try:
            nodes[node.get("id")] = [float(vals["xmin"]), float(vals["ymin"]),
                                     float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
    for e in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        if e.get("source") in nodes and e.get("target") in nodes:
            edges.add(frozenset((e.get("source"), e.get("target"))))
    return nodes, edges

def build_relation_examples(max_patches, pairs_per_patch):
    examples = []
    gml_files = sorted(PID2GRAPH_OPEN100.rglob("*.graphml"))
    rng.shuffle(gml_files)
    for gml in gml_files[:max_patches]:
        png = gml.with_suffix(".png")
        if not png.exists():
            continue
        try:
            nodes, edges = parse_graphml(gml)
        except ET.ParseError:
            continue
        if len(nodes) < 4 or not edges:
            continue
        node_ids = list(nodes)
        pos_pool = [tuple(e) for e in edges]
        rng.shuffle(pos_pool)
        neg_pool, attempts = [], 0
        while len(neg_pool) < pairs_per_patch and attempts < 200:
            attempts += 1
            a, b = rng.sample(node_ids, 2)
            if frozenset((a, b)) not in edges:
                neg_pool.append((a, b))
        for pair, connected in ([(p, True) for p in pos_pool[:pairs_per_patch // 2]] +
                                [(p, False) for p in neg_pool[:pairs_per_patch // 2]]):
            a, b = pair
            ba, bb = nodes[a], nodes[b]
            ux0, uy0 = min(ba[0], bb[0]), min(ba[1], bb[1])
            ux1, uy1 = max(ba[2], bb[2]), max(ba[3], bb[3])
            if ux1 - ux0 > MAX_PAIR_SPAN_PX or uy1 - uy0 > MAX_PAIR_SPAN_PX:
                continue
            cx0, cy0 = max(0, int(ux0 - CROP_MARGIN)), max(0, int(uy0 - CROP_MARGIN))
            cx1, cy1 = int(ux1 + CROP_MARGIN), int(uy1 + CROP_MARGIN)
            prompt = RELATION_PROMPT_TEMPLATE.format(
                ax0=int(ba[0] - cx0), ay0=int(ba[1] - cy0), ax1=int(ba[2] - cx0), ay1=int(ba[3] - cy0),
                bx0=int(bb[0] - cx0), by0=int(bb[1] - cy0), bx1=int(bb[2] - cx0), by1=int(bb[3] - cy0))
            target = ("Yes - these two symbols are directly connected." if connected
                      else "No - these two symbols are not directly connected.")
            examples.append({"task": "relation", "image_path": str(png), "crop": [cx0, cy0, cx1, cy1],
                             "prompt": prompt, "target_text": target})
    return examples

print("Building Gupta count + tag-read pools (Tesseract pass over 72 sheets - several minutes)...")
t0 = time.time()
count_examples, tag_examples = build_gupta_tasks()
print(f"  count={len(count_examples)} (zeros capped at ~{int(MAX_ZERO_COUNT_FRACTION*100)}%), "
      f"tag_read={len(tag_examples)} in {time.time()-t0:.0f}s")

kaggle_examples = build_kaggle_examples(MAX_KAGGLE_EXAMPLES)
relation_examples = build_relation_examples(MAX_RELATION_PATCHES, RELATION_PAIRS_PER_PATCH)
print(f"  typed={len(kaggle_examples)}, relation={len(relation_examples)}")

all_examples = count_examples + tag_examples + kaggle_examples + relation_examples
random.seed(0)
random.shuffle(all_examples)
print(f"\nTotal v2 pool: {len(all_examples)}", dict(Counter(e["task"] for e in all_examples)))
zero_frac = sum(1 for e in count_examples if " 0 " in e["target_text"]) / len(count_examples)
print(f"zero-count fraction in count task: {zero_frac:.2f} (target <= {MAX_ZERO_COUNT_FRACTION})")

## 6. Load Qwen3-VL-8B + fresh LoRA (from scratch — v1 adapter left untouched)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, get_peft_model

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
print("Base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Checkpoints (v2 path) + auto-resume

In [ ]:
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=CKPT_REPO, repo_type="model", private=True, exist_ok=True)

CKPT_LOCAL = Path("/content/checkpoints_v2")
LATEST_DIR = CKPT_LOCAL / CKPT_SUBDIR / "latest"
PROGRESS_FILE = LATEST_DIR / "progress.json"
CKPT_LOCAL.mkdir(exist_ok=True)

start_epoch, start_step = 0, 0
try:
    snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                      allow_patterns=[f"{CKPT_SUBDIR}/latest/*"], local_dir=str(CKPT_LOCAL))
except Exception as e:
    print("no remote snapshot pulled:", type(e).__name__)

if PROGRESS_FILE.exists() and (LATEST_DIR / "adapter_model.safetensors").exists():
    progress = json.loads(PROGRESS_FILE.read_text())
    start_epoch, start_step = progress["epoch"], progress["step"] + 1
    model.load_adapter(str(LATEST_DIR), adapter_name="default", is_trainable=True)
    print(f"RESUMED v2 from HF: epoch {start_epoch}, step {start_step}")
else:
    print("No v2 checkpoint - fresh start (correct for the first v2 run)")

def save_and_push_checkpoint(epoch, step, also_epoch_snapshot=False):
    model.save_pretrained(str(LATEST_DIR))
    PROGRESS_FILE.write_text(json.dumps({"epoch": epoch, "step": step}))
    try:
        api.upload_folder(folder_path=str(LATEST_DIR), path_in_repo=f"{CKPT_SUBDIR}/latest",
                          repo_id=CKPT_REPO, repo_type="model")
        if also_epoch_snapshot:
            api.upload_folder(folder_path=str(LATEST_DIR), path_in_repo=f"{CKPT_SUBDIR}/epoch_{epoch}",
                              repo_id=CKPT_REPO, repo_type="model")
    except Exception as e:
        print(f"  [warn] HF push failed ({type(e).__name__}) - local checkpoint kept, retry next interval")

## 8. Training loop — Run All with background execution, walk away

In [ ]:
import gc

def build_labeled_inputs(image, prompt, target_text, image_crop=None):
    if image_crop is not None:
        image = image.crop(image_crop)
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    prompt_len = prompt_inputs["input_ids"].shape[1]
    full = processor.apply_chat_template(
        messages + [{"role": "assistant", "content": [{"type": "text", "text": target_text}]}],
        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    labels = full["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full["labels"] = labels
    return {k: v.to(model.device) for k, v in full.items()}

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
run_t0 = time.time()
time_budget_s = TIME_BUDGET_HOURS * 3600
stopped_for_time = False
TASKS = ("count", "tag_read", "typed_summary", "relation")

for epoch in range(start_epoch, N_EPOCHS):
    random.seed(epoch)
    order = list(range(len(all_examples)))
    random.shuffle(order)
    skip_to = start_step if epoch == start_epoch else 0

    epoch_losses = {t: [] for t in TASKS}
    t0 = time.time()
    for step, idx in enumerate(order):
        if step < skip_to:
            continue
        if time.time() - run_t0 > time_budget_s:
            save_and_push_checkpoint(epoch, step - 1)
            print(f"TIME BUDGET REACHED - checkpointed at epoch {epoch} step {step-1}. Rerun to resume.")
            stopped_for_time = True
            break
        ex = all_examples[idx]
        try:
            img = Image.open(ex["image_path"]).convert("RGB")
            inputs = build_labeled_inputs(img, ex["prompt"], ex["target_text"], image_crop=ex.get("crop"))
            out = model(**inputs)
            loss = out.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            epoch_losses[ex["task"]].append(loss.item())
        except torch.cuda.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True)
            gc.collect(); torch.cuda.empty_cache()
            print(f"  [skip] step {step} OOM - cleared, continuing")
            continue
        except Exception as e:
            print(f"  [skip] step {step} ({ex['task']}): {type(e).__name__}: {e}")
            continue

        if step % 50 == 0:
            per_task = "  ".join(f"{t}={sum(v[-20:])/len(v[-20:]):.3f}" for t, v in epoch_losses.items() if v)
            print(f"epoch {epoch} step {step}/{len(order)} elapsed={time.time()-t0:.0f}s  {per_task}")
        if step % CHECKPOINT_EVERY_N_STEPS == 0 and step > 0:
            save_and_push_checkpoint(epoch, step)
            print(f"  [checkpoint] epoch {epoch} step {step}")

    if stopped_for_time:
        break
    save_and_push_checkpoint(epoch, len(order) - 1, also_epoch_snapshot=True)
    print(f"=== epoch {epoch} done (pushed {CKPT_SUBDIR}/epoch_{epoch}) ===")
    for task, losses in epoch_losses.items():
        if losses:
            print(f"  {task:15s} n={len(losses):5d} mean_loss={sum(losses)/len(losses):.4f}")
    start_step = 0

if not stopped_for_time:
    print(f"\nv2 training complete - adapter at {CKPT_REPO}/{CKPT_SUBDIR}")

## 9. After training — eval, not just smoke

Per-task loss lines during training now show each task separately (v1's single blended loss
hid the count/ocr failures). When done, run `Stage4_QwenDomainBase_Eval.ipynb` with
`ADAPTER_PATH_IN_REPO = "v2/latest"` for the real verdict against the same metrics as v1.
The quick generation check below is a sanity peek only.

In [ ]:
model.eval()
by_task = {}
for ex in all_examples:
    if ex["task"] not in by_task:
        by_task[ex["task"]] = ex
with torch.no_grad():
    for task, ex in by_task.items():
        img = Image.open(ex["image_path"]).convert("RGB")
        if ex.get("crop"):
            img = img.crop(ex["crop"])
        messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": ex["prompt"]}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=96, do_sample=False)
        text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"=== {task} ===")
        print("target:", ex["target_text"][:140])
        print("model :", text[:140])
        print()